# 13.04 · User interfaces

**Chapter:** Serving & Deployment · **Milestone:** M5

## Goals
- [ ] Build the same streaming chat UI in **Gradio** and **Streamlit**, both talking to the 13.03 API
- [ ] Understand each framework's execution model (Gradio: event handlers; Streamlit: rerun the script top to bottom)
- [ ] Test UIs programmatically (`gradio_client`, Streamlit `AppTest`)
- [ ] Know when to graduate: Chainlit, Open WebUI, and production front-ends (Next.js + Vercel AI SDK)

## Resources
- [Gradio – ChatInterface guide](https://www.gradio.app/guides/creating-a-chatbot-fast) — free
- [Streamlit – Build a chat app](https://docs.streamlit.io/develop/tutorials/chat-and-llm-apps/build-conversational-apps) · [AppTest](https://docs.streamlit.io/develop/api-reference/app-testing) — free
- [Chainlit](https://docs.chainlit.io/) · [Open WebUI](https://docs.openwebui.com/) — free/OSS
- [Vercel AI SDK](https://ai-sdk.dev/) (`useChat`, streaming UI in React/Next.js) — free/OSS

Architecture: **UI → API → LLM**. The UI never holds provider keys and never calls the model directly; it is a thin client of `app.py`.
That lets you swap UIs (Gradio today, React tomorrow) without touching the AI logic, and keeps rate limits/auth in one place.

| | Gradio | Streamlit | Chainlit | Open WebUI | Next.js + AI SDK |
|---|---|---|---|---|---|
| Best for | ML demos, HF Spaces | data apps + dashboards | chat/agent apps with step traces | ChatGPT-like UI for any OpenAI-compatible API | real products |
| Language | Python | Python | Python | (app, configure only) | TypeScript |
| Model | event handlers | rerun script each interaction | async callbacks | — | React components |

In [ ]:
import atexit
import subprocess
import sys
import time

import httpx

server = subprocess.Popen([sys.executable, "-m", "uvicorn", "app:app", "--port", "8765", "--log-level", "warning"])
atexit.register(server.terminate)  # stop it even if a later cell fails
for _ in range(120):
    try:
        print("API up:", httpx.get("http://localhost:8765/healthz").json()); break
    except httpx.ConnectError:
        time.sleep(0.5)

## 1. The shared SSE client
`ui_gradio.py` has `stream_answer(question)`, a generator that reads the API's SSE stream and yields `(text so far, source ids)`.
Both UIs reuse it. Try it directly:

In [ ]:
from ui_gradio import stream_answer

for text, sources in stream_answer("What is the capital of Uruguay?"):
    pass
print(text, "\nsources:", sources)

## 2. Gradio
`gr.ChatInterface(fn)` gives a full chat UI: `fn(message, history)` returns, or **yields** (streaming), the reply.
Every Gradio app is also an API (endpoint named after the function: `/respond`): `gradio_client` calls it like a function — handy for tests.
In the notebook we launch without blocking; from the terminal run `uv run python ui_gradio.py` and open http://localhost:7860.

In [ ]:
import ui_gradio
from gradio_client import Client

ui_gradio.demo.launch(server_port=7861, prevent_thread_lock=True, quiet=True)
reply = Client("http://127.0.0.1:7861/", verbose=False).predict(message="Who was Abraham Lincoln?", api_name="/respond")
print(reply[:500])
ui_gradio.demo.close()

Gradio niceties: `gr.Blocks` for custom layouts, multimodal input (`multimodal=True` for images/audio, 14.x), `share=True` for a temporary
public link, and one-click hosting on **Hugging Face Spaces** (13.05).

## 3. Streamlit
Streamlit's model: **the whole script reruns on every interaction**. Anything that must survive a rerun goes in `st.session_state`;
expensive objects go in `@st.cache_resource`. Read `ui_streamlit.py` (~30 lines). Run it with `uv run streamlit run ui_streamlit.py`.

`AppTest` runs the script headless and lets you type into widgets:

In [ ]:
from streamlit.testing.v1 import AppTest

from pathlib import Path

# AppTest resolves relative paths against the calling *file*; in a notebook that is a temp file, so pass an absolute path
at = AppTest.from_file(str(Path("ui_streamlit.py").resolve()), default_timeout=90).run()
at.chat_input[0].set_value("Who was Abraham Lincoln?").run()
print("messages:", [m.name for m in at.chat_message])
print(at.chat_message[-1].markdown[0].value[:400])
print(at.chat_message[-1].caption[0].value if at.chat_message[-1].caption else "(no sources)")

In [ ]:
server.terminate()
_ = server.wait()

## 4. Ready-made chat front-ends
**Open WebUI** is a ChatGPT-style app (accounts, history, RAG uploads, tools) for any OpenAI-compatible endpoint, Ollama included:
```bash
uvx --python 3.11 open-webui serve        # http://localhost:8080 ; detects Ollama automatically
```
Point it at your own API by adding an OpenAI-compatible `/v1/chat/completions` route to `app.py` (exercise 1 of 13.03).

**Chainlit** shows agent steps (tool calls, retrieval) as collapsible elements, which is great for 09-style agents:
```python
import chainlit as cl
@cl.on_message
async def main(message: cl.Message):
    async with cl.Step(name="retrieve") as step:
        step.output = rag.search(message.content)
    await cl.Message(content=...).send()
```

## 5. Production front-ends: Next.js + Vercel AI SDK
For a real product, a TypeScript front-end with the AI SDK's `useChat` hook handles streaming, message state, tool-call rendering and
abort/retry. It can call providers directly from a Next.js route, or your Python API:
```tsx
'use client';
import { useChat } from '@ai-sdk/react';
export default function Chat() {
  const { messages, sendMessage, status } = useChat();   // POSTs to /api/chat and streams the reply
  return (<div>{messages.map(m => <p key={m.id}>{m.role}: {m.parts.map(p => p.type === 'text' ? p.text : '').join('')}</p>)}
    <form onSubmit={e => { e.preventDefault(); sendMessage({ text: e.currentTarget.q.value }); }}><input name="q" /></form></div>);
}
```
Other options: assistant-ui, CopilotKit, or plain `fetch` + `ReadableStream` against your SSE endpoint.

**UX checklist for LLM apps**: stream tokens, show sources, let users stop generation, show errors in plain words, keep conversation history,
collect 👍/👎 feedback (11.05), and make latency visible ("searching…", "thinking…").

## Exercises
1. Add 👍/👎 buttons to the Gradio UI (`gr.ChatInterface(..., flagging_mode="manual")` or a `.like()` event on `gr.Chatbot`) and log them to
   a JSONL file in the 11.05 format.
2. Add a sidebar to the Streamlit app with a `k` slider (number of retrieved passages) and show the sources in an expander.
3. Run Open WebUI against Ollama and compare its UX with your Gradio app. What would it take to make yours as usable?
4. (Optional, JS) Build the Next.js + AI SDK version that calls your `/v1/ask/stream` endpoint.

In [ ]:
# your code here